# Constrained n=200 dynamic windfield — A100, **saved to Google Drive**

Runs our PyTorch solver (MUSCL-fixed) on the A100 and writes **all outputs to Google Drive**, so a Colab 
disconnect/reset cannot lose your work. Every finished batch is a checkpoint on Drive; if the runtime dies, 
just re-run the setup + run cells and it **resumes** from where it stopped.

**Before running:** `Runtime → Change runtime type → A100 GPU`.

**You upload `colab_bundle.zip` once** (the first time). After that it lives on Drive and reconnects skip the upload.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Runtime to A100')

## 2. Mount Drive & set up the working dir on Drive
First run: pick `colab_bundle.zip` when prompted. Reconnecting later: it detects the existing Drive copy and **resumes** (no re-upload).

In [ ]:
MM = '/content/drive/MyDrive/mm_dynamic'
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
if os.path.exists(os.path.join(MM, 'pipeline', 'windfield_dynamic_batch.py')):
    print('Working dir already on Drive — will RESUME:', MM)
    ck = os.path.join(MM, 'outputs/dynamic/precompute_constrained')
    n = len([f for f in os.listdir(ck) if f.endswith('.json')]) if os.path.exists(ck) else 0
    print(f'  {n} batch checkpoint(s) already saved on Drive.')
else:
    os.makedirs(MM, exist_ok=True)
    from google.colab import files
    up = files.upload()  # choose colab_bundle.zip
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall(MM)
    print('bundle extracted to Drive:', MM)

## 3. Choose what to run
- `''` or `'all'` — all 200 storms (cost-sorted, cheap first)
- `'bulk'` — the cheap 155 · `'tail'` — the expensive 45 · `'vec:75,90'` — specific storms

`BATCH_SIZE`: storms marched together. 25 is safe; on the A100's 40 GB you can try 100–155 for the 
latency-amortization speedup (keep batches cost-homogeneous — use with `'bulk'` or `'tail'`, not all 200).

In [ ]:
SELECT = 'all'      # '' | 'all' | 'bulk' | 'tail' | 'vec:75,90'
BATCH_SIZE = 25     # 25 default; try 100-155 on A100 with SELECT='bulk'

## 4. Run (writes to Drive as it goes)
Streams progress. **If it disconnects:** reconnect, re-run cells 1–2, then this cell — it skips finished batches and resumes.

In [ ]:
MM = '/content/drive/MyDrive/mm_dynamic'
import subprocess, sys
cmd = [sys.executable, '-u', 'pipeline/windfield_dynamic_batch.py',
       '--constrained', '--select', SELECT, '--batch-size', str(BATCH_SIZE)]
print('running:', ' '.join(cmd), '\n')
p = subprocess.Popen(cmd, cwd=MM, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    if not any(w in line for w in ('Warning','warnings.warn','ConvergenceWarning')):
        print(line, end='')
p.wait()
print('\nexit code:', p.returncode)

## 5. Assemble & save results (to Drive **and** download)
Safe to run anytime — builds products from whatever checkpoints exist (partial runs OK), saves a zip to Drive, and downloads it to your computer.

In [ ]:
MM = '/content/drive/MyDrive/mm_dynamic'
import subprocess, sys, glob, os, zipfile
subprocess.run([sys.executable, 'pipeline/windfield_dynamic_batch.py',
                '--constrained', '--assemble-only'], cwd=MM)
out = '/content/drive/MyDrive/dynamic_constrained_results.zip'
prod = glob.glob(f'{MM}/outputs/web/powell_dyn_constrained*.json')
ckpt = glob.glob(f'{MM}/outputs/dynamic/precompute_constrained/*.json')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in prod + ckpt:
        z.write(f, os.path.relpath(f, MM))
print(f'{len(prod)} products + {len(ckpt)} checkpoints -> {out} (also on Drive)')
from google.colab import files
files.download(out)

### If the runtime disconnects while you're away
Nothing is lost — the finished batches are on Drive. On return: re-run **Cell 1**, **Cell 2** (it resumes, no upload), 
then **Cell 4** to finish, or **Cell 5** to grab what's done. The `dynamic_constrained_results.zip` also stays on Drive.